# Fine-tuning d’un SLM sur un dataset de classification

## Pourquoi fine-tuner un SLM (Small Language Model) plutôt qu'un LLM ?

## Avantages des SLMs

1. **Ressources computationnelles réduites**
   - Les SLMs nécessitent moins de puissance de calcul
   - Temps d'entraînement plus court
   - Coûts d'infrastructure plus faibles

2. **Meilleure interprétabilité**
   - Architecture plus simple et plus facile à comprendre
   - Plus facile d'analyser les décisions du modèle
   - Meilleur contrôle sur le processus d'apprentissage

3. **Spécialisation sur des tâches spécifiques**
   - Performance potentiellement supérieure sur des tâches ciblées
   - Moins de bruit et de connaissances non pertinentes
   - Adaptation plus précise au domaine d'application

## Étapes du fine-tuning d'un SLM

1. **Préparation des données**
   - Collecte du dataset spécifique au domaine
   - Nettoyage et prétraitement des données
   - Division en ensembles d'entraînement/validation/test

2. **Configuration du modèle**
   - Choix de l'architecture de base appropriée
   - Définition des hyperparamètres
   - Mise en place des techniques d'optimisation (QLora, PEFt)

3. **Processus d'entraînement**
   - Fine-tuning progressif
   - Monitoring des métriques
   - Validation régulière des performances

4. **Évaluation et optimisation**
   - Tests sur données de validation
   - Ajustements des hyperparamètres si nécessaire
   - Vérification des performances sur cas d'usage réels


# SLMs (Small Language Models) populaires

| Modèle | Taille | Description | Cas d'usage typiques |
|--------|---------|-------------|---------------------|
| BERT-small | 66M | Version réduite de BERT | Classification de texte, NER |
| DistilBERT | 66M | Version distillée de BERT | Analyse de sentiment, QA |
| TinyBERT | 14.5M | Version très compacte de BERT | Applications mobiles |
| MiniLM | 22M | Architecture optimisée | Embeddings, Classification |
| ALBERT | 12M-18M | Architecture légère avec partage de paramètres | NLU, Classification |
| RoBERTa-small | 84M | Version optimisée de BERT | NLP général |
| GPT-2-small | 124M | Plus petit modèle GPT-2 | Génération de texte |
| T5-small | 60M | Version réduite de T5 | Traduction, Résumé |


#  Configuration initiale

In [1]:
# Install necessary libraries
!pip install transformers datasets evaluate scikit-learn


  Using cached transformers-4.56.0-py3-none-any.whl.metadata (40 kB)
  Using cached datasets-4.0.0-py3-none-any.whl.metadata (19 kB)
  Using cached evaluate-0.4.5-py3-none-any.whl.metadata (9.5 kB)
  Using cached tokenizers-0.22.0-cp39-abi3-macosx_11_0_arm64.whl.metadata (6.8 kB)
  Using cached safetensors-0.6.2-cp38-abi3-macosx_11_0_arm64.whl.metadata (4.1 kB)
  Using cached xxhash-3.5.0-cp313-cp313-macosx_11_0_arm64.whl.metadata (12 kB)
  Using cached multiprocess-0.70.16-py312-none-any.whl.metadata (7.2 kB)
  Using cached fsspec-2025.3.0-py3-none-any.whl.metadata (11 kB)
Using cached transformers-4.56.0-py3-none-any.whl (11.6 MB)
Using cached tokenizers-0.22.0-cp39-abi3-macosx_11_0_arm64.whl (2.9 MB)
Using cached datasets-4.0.0-py3-none-any.whl (494 kB)
Using cached fsspec-2025.3.0-py3-none-any.whl (193 kB)
Using cached multiprocess-0.70.16-py312-none-any.whl (146 kB)
Using cached evaluate-0.4.5-py3-none-any.whl (84 kB)
Using cached safetensors-0.6.2-cp38-abi3-macosx_11_0_arm64.whl 

In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


# Charger la base de données 


## Problématique
Le phishing est une technique de cybercriminalité qui consiste à créer des sites web
imitant des sites légitimes (banques, réseaux sociaux, etc.) pour voler les données
personnelles des utilisateurs. La détection automatique de ces sites est cruciale
pour protéger les internautes.

## Description du dataset
- Source: https://huggingface.co/datasets/shawhin/phishing-site-classification
- Contenu: ~10K exemples d'URLs
- Features: 
  * URLs complètes des sites
  * Caractéristiques extraites (longueur, présence de caractères suspects, etc.)
  * Domaines et sous-domaines
- Labels: 
  * 0 = Site légitime
  * 1 = Site de phishing

## Objectif
Entraîner un modèle pour classifier automatiquement les URLs suspectes



In [3]:
# 1. Charger le dataset (phishing)
dataset = load_dataset("shawhin/phishing-site-classification")
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'labels'],
        num_rows: 2100
    })
    validation: Dataset({
        features: ['text', 'labels'],
        num_rows: 450
    })
    test: Dataset({
        features: ['text', 'labels'],
        num_rows: 450
    })
})

# Tokenizer

# # 2. Tokenizer
 Utilisation de DistilBERT comme modèle de base pour le tokenizer
DistilBERT est une version plus légère et plus rapide de BERT
 tout en conservant 97% de ses performances


In [4]:
model_id = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Importer le modèle

In [5]:
id2label = {0: "Sécurisé", 1: "Dangereux"}
label2id = {"Sécurisé": 0, "Dangereux": 1}
model = AutoModelForSequenceClassification.from_pretrained(model_id,
                                                        num_labels=2,
                                                        id2label=id2label,
                                                        label2id=label2id,
                                                      )

ImportError: 
AutoModelForSequenceClassification requires the PyTorch library but it was not found in your environment. Check out the instructions on the
installation page: https://pytorch.org/get-started/locally/ and follow the ones that match your environment.
Please note that you may need to restart your runtime after installation.


# Gel du modèle de base (Freeze base model)

 Pour optimiser l'entraînement et éviter le surapprentissage:
 - Les poids du modèle de base DistilBERT sont gelés
 - Seules les couches de classification ajoutées seront entraînées
 - Cela permet de:
   * Réduire le nombre de paramètres à entraîner
   * Préserver les connaissances générales du modèle pré-entraîné
   * Accélérer l'entraînement
   * Économiser la mémoire GPU
  * Éviter l'oubli catastrophique des connaissances de base


In [ ]:
# print layers
for name, param in model.named_parameters():
   print(f"Couche: {name}, Entraînable: {param.requires_grad}")

Couche: distilbert.embeddings.word_embeddings.weight, Entraînable: True
Couche: distilbert.embeddings.position_embeddings.weight, Entraînable: True
Couche: distilbert.embeddings.LayerNorm.weight, Entraînable: True
Couche: distilbert.embeddings.LayerNorm.bias, Entraînable: True
Couche: distilbert.transformer.layer.0.attention.q_lin.weight, Entraînable: True
Couche: distilbert.transformer.layer.0.attention.q_lin.bias, Entraînable: True
Couche: distilbert.transformer.layer.0.attention.k_lin.weight, Entraînable: True
Couche: distilbert.transformer.layer.0.attention.k_lin.bias, Entraînable: True
Couche: distilbert.transformer.layer.0.attention.v_lin.weight, Entraînable: True
Couche: distilbert.transformer.layer.0.attention.v_lin.bias, Entraînable: True
Couche: distilbert.transformer.layer.0.attention.out_lin.weight, Entraînable: True
Couche: distilbert.transformer.layer.0.attention.out_lin.bias, Entraînable: True
Couche: distilbert.transformer.layer.0.sa_layer_norm.weight, Entraînable: True

In [ ]:
# geler les paramètres du modèle de base
for name, param in model.base_model.named_parameters():
    param.requires_grad = False

# dégeler les couches de pooling du modèle de base 
for name, param in model.base_model.named_parameters():
    if "pooler" in name:
        param.requires_grad = True


In [ ]:
# print layers
for name, param in model.named_parameters():
   print(name, param.requires_grad)

distilbert.embeddings.word_embeddings.weight False
distilbert.embeddings.position_embeddings.weight False
distilbert.embeddings.LayerNorm.weight False
distilbert.embeddings.LayerNorm.bias False
distilbert.transformer.layer.0.attention.q_lin.weight False
distilbert.transformer.layer.0.attention.q_lin.bias False
distilbert.transformer.layer.0.attention.k_lin.weight False
distilbert.transformer.layer.0.attention.k_lin.bias False
distilbert.transformer.layer.0.attention.v_lin.weight False
distilbert.transformer.layer.0.attention.v_lin.bias False
distilbert.transformer.layer.0.attention.out_lin.weight False
distilbert.transformer.layer.0.attention.out_lin.bias False
distilbert.transformer.layer.0.sa_layer_norm.weight False
distilbert.transformer.layer.0.sa_layer_norm.bias False
distilbert.transformer.layer.0.ffn.lin1.weight False
distilbert.transformer.layer.0.ffn.lin1.bias False
distilbert.transformer.layer.0.ffn.lin2.weight False
distilbert.transformer.layer.0.ffn.lin2.bias False
distilbe

# Preprocessing du texte

In [ ]:
def tokenize(example):
    return tokenizer(example["text"], truncation=True, padding="max_length", max_length=128)

tokenized_dataset = dataset.map(tokenize, batched=True)


Map:   0%|          | 0/450 [00:00<?, ? examples/s]

In [ ]:
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 2100
    })
    validation: Dataset({
        features: ['text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 450
    })
    test: Dataset({
        features: ['text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 450
    })
})

# Préparer les métriques

In [ ]:
# Import evaluate library
import evaluate
import numpy as np
# Chargement des métriques à évaluer
accuracy = evaluate.load("accuracy")       # Pour le calcul du score de précision (accuracy)
auc_score = evaluate.load("roc_auc")       # Pour le calcul de l’AUC (aire sous la courbe ROC)

def compute_metrics(eval_pred):
    # Récupération des prédictions et des vraies étiquettes
    predictions, labels = eval_pred

    # Application de la fonction softmax pour obtenir les probabilités
    probabilities = np.exp(predictions) / np.exp(predictions).sum(-1, keepdims=True)

    # Extraction des probabilités de la classe positive (index 1)
    positive_class_probs = probabilities[:, 1]

    # Calcul de l’AUC à partir des probabilités et des étiquettes vraies
    auc = float(round(
        auc_score.compute(prediction_scores=positive_class_probs, references=labels)['roc_auc'], 3)
    )

    # Prédiction de la classe la plus probable
    predicted_classes = np.argmax(predictions, axis=1)

    # Calcul de l’accuracy
    acc = float(round(
        accuracy.compute(predictions=predicted_classes, references=labels)['accuracy'], 3)
    )

    # Retour des métriques sous forme de dictionnaire
    return {"Accuracy": acc, "AUC": auc}



In [ ]:
# 5. Configuration d'entraînement
from transformers import TrainingArguments, Trainer

#  Définition des hyperparamètres d'entraînement
lr = 2e-4              # Taux d’apprentissage (learning rate)
batch_size = 8         # Taille du lot (batch size) pour l’entraînement et l’évaluation
num_epochs = 10        # Nombre total d’époques d'entraînement

#  Configuration des paramètres d'entraînement avec Hugging Face
training_args = TrainingArguments(
    output_dir="bert-phishing-classifier",      # Dossier de sortie pour sauvegarder le modèle et les checkpoints
    learning_rate=lr,                           # Taux d’apprentissage utilisé par l’optimiseur
    per_device_train_batch_size=batch_size,     # Taille du batch par GPU pour l’entraînement
    per_device_eval_batch_size=batch_size,      # Taille du batch par GPU pour l’évaluation
    num_train_epochs=num_epochs,                # Nombre total d’époques d'entraînement
    save_strategy="epoch",                      # Sauvegarder le modèle à la fin de chaque époque
    eval_strategy="epoch",                      # Évaluer le modèle à la fin de chaque époque
    eval_steps=1,                               # [⚠️ Option redondante si `eval_strategy='epoch'`] Fréquence des évaluations (en nombre de steps)
    metric_for_best_model="AUC",                # Métrique utilisée pour sélectionner le meilleur modèle
    greater_is_better=True,                     # Indique que les valeurs plus élevées d’AUC sont meilleures
    load_best_model_at_end=True,                # Recharge automatiquement le meilleur modèle à la fin de l’entraînement
)


In [ ]:
# 6. Entrainement

from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


trainer = Trainer(
    model=model,                                    # Le modèle à entraîner
    args=training_args,                            # Arguments d'entraînement définis précédemment
    train_dataset=tokenized_dataset["train"],      # Dataset d'entraînement tokenisé
    eval_dataset=tokenized_dataset["test"],        # Dataset de test pour l'évaluation
    data_collator=data_collator,                   # Fonction pour regrouper les données en batch
    compute_metrics=compute_metrics,               # Fonction de calcul des métriques
    tokenizer=tokenizer                           # Tokenizer utilisé pour le prétraitement
)

trainer.train()                                    # Lance l'entraînement du modèle

/var/folders/_h/9jm4l3w973n8rgwzstgvzkzc0000gn/T/ipykernel_34390/781016876.py:8: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/opt/anaconda3/envs/genai/lib/python3.13/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy,Auc
1,No log,0.380899,0.813000,0.911000
2,0.415700,0.325404,0.833000,0.936000
3,0.415700,0.319479,0.853000,0.942000
4,0.319100,0.326987,0.856000,0.944000
5,0.319100,0.295960,0.873000,0.949000
6,0.297600,0.292660,0.876000,0.950000
7,0.297600,0.289282,0.873000,0.952000
8,0.282200,0.287441,0.878000,0.952000
9,0.282200,0.285719,0.887000,0.952000
10,0.282000,0.285273,0.880000,0.953000


/opt/anaconda3/envs/genai/lib/python3.13/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/opt/anaconda3/envs/genai/lib/python3.13/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/opt/anaconda3/envs/genai/lib/python3.13/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/opt/anaconda3/envs/genai/lib/python3.13/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/opt/anaconda3/envs/genai/lib/python3.13/site-pa

TrainOutput(global_step=2630, training_loss=0.3166541487545115, metrics={'train_runtime': 169.4469, 'train_samples_per_second': 123.933, 'train_steps_per_second': 15.521, 'total_flos': 695453842944000.0, 'train_loss': 0.3166541487545115, 'epoch': 10.0})

In [ ]:
# Évaluation du modèle entraîné
metrics = trainer.evaluate()

# Affichage clair des résultats
print("Résultats de l'évaluation finale :")
for key, value in metrics.items():
    print(f"{key}: {round(value, 4)}")


/opt/anaconda3/envs/genai/lib/python3.13/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Résultats de l'évaluation finale :
eval_loss: 0.2853
eval_Accuracy: 0.88
eval_AUC: 0.953
eval_runtime: 2.7114
eval_samples_per_second: 165.964
eval_steps_per_second: 21.022
epoch: 10.0


# Appliquer le modèle au jeu de données de test

In [ ]:
predictions = trainer.predict(tokenized_dataset["test"])

# Extraire les logits et les étiquettes de l'objet predictions
logits = predictions.predictions
labels = predictions.label_ids

# Utiliser la fonction compute_metrics
metrics = compute_metrics((logits, labels))
print("Résultats détaillés de l'évaluation :")
print(metrics)

/opt/anaconda3/envs/genai/lib/python3.13/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Résultats détaillés de l'évaluation :
{'Accuracy': 0.88, 'AUC': 0.953}
